# MNIST Classification

## Import packages

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
import os
from pathlib import Path

## Set up hyperparameter

In [ ]:
batch_size = 64
test_batch_size = 1000
epochs = 10
lr = 0.01
try_cuda = True
seed = 42
logging_interval = 10 # how many batches to wait before logging
logging_dir = None

## Set up our Neural Network

In [ ]:
# Defining Architecture,loss and optimizer
class Net(nn.Module):
    def __init__(self):

        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=5)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5)
        self.drop = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(1024, 1024)
        self.fc2 = nn.Linear(1024, 10)

    def forward(self, x):

        # conv1(5-5-1-32) -> ReLU -> maxpool(2-2)
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        
        # conv2(5-5-32-64) -> ReLU -> maxpool(2-2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        
        # from 64*4*4 to 1024
        x = x.view(-1, 1024)
        
        # fc(1024) -> ReLU -> DropOut
        x = self.fc1(x)
        x = F.relu(x)
        x = self.drop(x)
        
        # fc -> softmax
        x = self.fc2(x)
        x = F.softmax(x, dim=1)


        return x
    


## Instantiate our NN

In [ ]:

model = Net()
optimizer = optim.Adam(model.parameters(), lr=lr)
writer = SummaryWriter(log_dir=logging_dir)
# Defining the test and trainig loops
eps=1e-13

## Set up Enviorment

In [ ]:
#deciding whether to send to the cpu or not if available
if torch.cuda.is_available() and try_cuda:
    cuda = True
    torch.cuda.manual_seed(seed)
    model = model.cuda()
else:
    cuda = False
    torch.manual_seed(seed)

# Setting up data
transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.01307,), (0.3081,))
])

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('../data', train=True, download=True,
                   transform=transform),
    batch_size=batch_size, shuffle=True)

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('../data', train=False, transform=transform),
    batch_size=batch_size, shuffle=True)

## Define a Training loop

In [ ]:
def train(epoch):
    model.train()

    #criterion = nn.CrossEntropyLoss()
    criterion = nn.NLLLoss(reduction='none')

    for batch_idx, (data, target) in enumerate(train_loader):
        if cuda:
            data, target = data.cuda(), target.cuda()

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(torch.log(output+eps), target) # = sum_k(-t_k * log(y_k))
        loss = loss.mean() # average over the batch
        loss.backward()
        optimizer.step()

        if batch_idx % logging_interval == 0:
            # [inset-code: print and log the performance]
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} ({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')

    # Log model parameters to TensorBoard at every epoch
    for name, param in model.named_parameters():
        layer, attr = os.path.splitext(name)
        attr = attr[1:]
        writer.add_histogram(f'{layer}/{attr}',param.clone().cpu().data.numpy(), epoch)

## Define a Testing loop


In [ ]:
def test(epoch):
    model.eval()
    test_loss = 0
    correct = 0
    #criterion = nn.CrossEntropyLoss()

    #criterion = nn.CrossEntropyLoss(size_average = False)
    criterion = nn.NLLLoss(reduction='none')

    for data, target in test_loader:
        if cuda:
            data, target = data.cuda(), target.cuda()

        output = model(data)

        test_loss += criterion(torch.log(output+eps), target).sum().item() # sum up batch loss (later, averaged over all test samples)
        pred = output.argmax(dim=1) # get the index of the max log-probability
        correct += pred.eq(target).sum().item()

    test_loss /= len(test_loader.dataset)
    test_accuracy = 100. * correct / len(test_loader.dataset)
    print(f'Test Epoch: {epoch}\tLoss: {test_loss:.6f}\tAccuracy: {test_accuracy:.2f}%')

    # Log test/loss and test/accuracy to TensorBoard at every epoch
    n_iter = epoch * len(train_loader)
    writer.add_scalar('test/loss', test_loss, n_iter)
    writer.add_scalar('test/accuracy', test_accuracy, n_iter)

## Run those loops

In [ ]:

training_start_time = datetime.now()
for epoch in range(1, epochs + 1):
    train(epoch)
    test(epoch)

writer.close()